[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/06_02_main_cnn.ipynb)

# Module 6, Vision: Letting the Network Learn the Features

**Notebook:** `06_02_main_cnn`

## Where this fits

In `06_01`, we converted each satellite image into 34 measurements that **we chose**:

\[
\text{image}
\rightarrow
\text{human-designed features}
\rightarrow
\text{classifier}
\]

Now we remove that feature-engineering step.

We give neural networks the pixels and ask them to learn useful representations themselves.

We will train two models on the **same EuroSAT sample and the same train/validation/test split**:

### Model A — Dense network

Flatten the `64 × 64 × 3` image into 12,288 numbers and feed them into ordinary dense layers.

The model sees every pixel, but the architecture has no built-in notion that nearby pixels are related.

### Model B — Convolutional neural network

Keep the image's spatial structure and apply learned local filters across the image.

The CNN builds directly on the intuition from `06_00`:

- local filters;
- feature maps;
- downsampling;
- increasingly abstract representations.

## The key question

> **If both models receive the same pixels, does an architecture designed for spatial data learn a better representation?**

## 0) Setup

This notebook is designed to run independently in Google Colab.

TensorFlow is the main dependency. A GPU helps, but this small experiment can also run on CPU.

We use the same deterministic sample and 70/10/20 split logic as `06_01_main_classical`.

In [ ]:
import sys
import subprocess
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

import tensorflow as tf

if importlib.util.find_spec("datasets") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "datasets"]
    )

from datasets import load_dataset, concatenate_datasets

SEED = 1955
N_SAMPLES = 5000
BATCH_SIZE = 64

tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))


## 1) Load the same EuroSAT sample

The data logic below intentionally matches `06_01`:

1. load EuroSAT;
2. sort all image paths;
3. draw the same seeded sample of 5,000 images;
4. create the same 70/10/20 train/validation/test split.

That makes the comparisons across the vision notebooks as close to apples-to-apples as possible.

In [ ]:
HF_DATASET = "giswqs/EuroSAT_RGB"

print("Loading EuroSAT from Hugging Face...")
dataset_dict = load_dataset(HF_DATASET)

# Combine the hosted splits because we create our own deterministic course split.
full_dataset = concatenate_datasets(
    [
        dataset_dict["train"],
        dataset_dict["validation"],
        dataset_dict["test"],
    ]
).sort("filename")

label_names = list(
    full_dataset.features["label"].names
)
num_classes = len(label_names)

print(f"Hosted images: {len(full_dataset):,}")
print("Classes:", label_names)

# Deterministic sample from filenames sorted above.
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(
    len(full_dataset),
    size=N_SAMPLES,
    replace=False,
)

sampled = full_dataset.select(
    sample_idx.tolist()
)

y = np.asarray(
    sampled["label"],
    dtype=np.int64,
)

idx_all = np.arange(N_SAMPLES)

idx_dev, idx_test = train_test_split(
    idx_all,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

idx_train, idx_val = train_test_split(
    idx_dev,
    test_size=0.125,
    stratify=y[idx_dev],
    random_state=SEED,
)

X_img = np.empty(
    (N_SAMPLES, 64, 64, 3),
    dtype=np.uint8,
)

for i, row in enumerate(sampled):
    X_img[i] = np.asarray(
        row["image"].convert("RGB"),
        dtype=np.uint8,
    )

print(
    f"train={len(idx_train):,}  "
    f"validation={len(idx_val):,}  "
    f"test={len(idx_test):,}"
)
print("Image array:", X_img.shape, X_img.dtype)

X_train = X_img[idx_train]
X_val = X_img[idx_val]
X_test = X_img[idx_test]

y_train = y[idx_train]
y_val = y[idx_val]
y_test = y[idx_test]

print("train:", X_train.shape)
print("validation:", X_val.shape)
print("test:", X_test.shape)


### Teaching note about the test set

We inspect test performance throughout this module because our goal is to compare approaches interactively.

In a formal modeling project, repeatedly using test results to choose the next model would make the test set part of the development process. A separate untouched final evaluation set would normally be retained.

## 2) Build the TensorFlow input pipeline

Neural networks work best with batched numerical tensors.

We will:

- convert pixels from integers to `float32`;
- scale values from `0–255` to `0–1`;
- shuffle the training data;
- batch and prefetch for efficiency.

No hand-designed visual features are created.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(images, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (images, labels)
    )

    ds = ds.map(
        lambda x, y: (
            tf.cast(x, tf.float32) / 255.0,
            y,
        ),
        num_parallel_calls=AUTOTUNE,
    )

    if shuffle:
        ds = ds.shuffle(
            2048,
            seed=SEED,
        )

    return (
        ds
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )

ds_train = make_dataset(
    X_train,
    y_train,
    shuffle=True,
)

ds_val = make_dataset(
    X_val,
    y_val,
)

ds_test = make_dataset(
    X_test,
    y_test,
)

IMG_SHAPE = (64, 64, 3)

for images, labels_batch in ds_train.take(1):
    print("image batch:", images.shape, images.dtype)
    print("label batch:", labels_batch.shape, labels_batch.dtype)

## 3) Make the comparison fair

We want the main difference between the two networks to be **architecture**, not training privilege.

So both models will receive:

- the same images;
- the same train/validation/test split;
- the same random flip augmentation;
- Adam with the same initial learning rate;
- the same maximum number of epochs;
- the same learning-rate reduction;
- the same early-stopping logic.

This does not make the models identical in every respect, but it makes the comparison much more informative than giving one model far more training or regularization than the other.

In [ ]:
def make_callbacks():
    return [
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

MAX_EPOCHS = 50

## 4) Model A: flatten the pixels

A `64 × 64 × 3` image contains:

\[
64 \times 64 \times 3 = 12{,}288
\]

pixel values.

The dense baseline simply flattens the image into a 12,288-element vector.

That creates an important limitation:

> After flattening, the architecture does not explicitly know that pixel 101 was next to pixel 102 in the original image.

A dense layer can still learn patterns from the training data, but it does not build in the assumptions of **locality** and **weight sharing** that make CNNs well suited to images.

In [ ]:
dense_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=IMG_SHAPE),

        # Same augmentation used by the CNN.
        tf.keras.layers.RandomFlip(
            "horizontal_and_vertical",
            seed=SEED,
        ),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(
            256,
            activation="relu",
        ),

        tf.keras.layers.Dense(
            128,
            activation="relu",
        ),

        tf.keras.layers.Dense(
            num_classes,
            activation="softmax",
        ),
    ],
    name="dense_baseline",
)

dense_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

dense_model.summary()

In [ ]:
history_dense = dense_model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=MAX_EPOCHS,
    callbacks=make_callbacks(),
    verbose=2,
)

## 5) Model B: preserve spatial structure with convolution

Now we use the architectural ideas from `06_00`.

### Convolution

A `3 × 3` filter examines a local neighborhood.

Unlike the Sobel filter we specified by hand, these filter weights start from random values and are updated through gradient descent.

### Weight sharing

The **same filter** is applied across the image.

A filter that learns to respond to a useful pattern can detect that pattern in many locations without learning a separate set of weights for every position.

### Multiple filters

`Conv2D(32, 3)` means:

> Learn 32 different `3 × 3` filters.

Each filter creates its own feature map.

### Downsampling

After each convolutional block, max pooling reduces the spatial resolution:

\[
64 \rightarrow 32 \rightarrow 16 \rightarrow 8
\]

while increasing the number of learned feature channels:

\[
3 \rightarrow 32 \rightarrow 64 \rightarrow 128
\]

The representation becomes spatially smaller but semantically richer.

In [ ]:
cnn_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=IMG_SHAPE),

        tf.keras.layers.RandomFlip(
            "horizontal_and_vertical",
            seed=SEED,
        ),

        tf.keras.layers.Conv2D(
            32,
            kernel_size=3,
            padding="same",
            activation="relu",
            name="conv1",
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(
            64,
            kernel_size=3,
            padding="same",
            activation="relu",
            name="conv2",
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(
            128,
            kernel_size=3,
            padding="same",
            activation="relu",
            name="conv3",
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.GlobalAveragePooling2D(),

        tf.keras.layers.Dense(
            128,
            activation="relu",
        ),

        tf.keras.layers.Dropout(
            0.25,
            seed=SEED,
        ),

        tf.keras.layers.Dense(
            num_classes,
            activation="softmax",
        ),
    ],
    name="cnn_small",
)

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_model.summary()

### Look at the parameter counts

The dense model usually has **far more trainable parameters** than this small CNN.

That makes the comparison useful:

> A model can have more parameters and still be a worse fit for the structure of the problem.

The CNN's advantage is not simply “more capacity.”

It is the assumptions built into the architecture:

- local connectivity;
- shared filters;
- hierarchical feature construction.

In [ ]:
dense_params = dense_model.count_params()
cnn_params = cnn_model.count_params()

print(f"Dense parameters: {dense_params:,}")
print(f"CNN parameters  : {cnn_params:,}")
print()
print(f"Dense / CNN parameter ratio: {dense_params / cnn_params:.1f}×")

In [ ]:
history_cnn = cnn_model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=MAX_EPOCHS,
    callbacks=make_callbacks(),
    verbose=2,
)

## 6) Compare the learning curves

Do not look only at the final accuracy.

Ask:

- How quickly does each model learn?
- When does validation performance stop improving?
- Is there a large train/validation gap?
- Does one architecture appear to use the available data more efficiently?

The curves help us separate **learning behavior** from a single final score.

In [ ]:
fig, axs = plt.subplots(
    1,
    2,
    figsize=(13, 4),
)

for ax, history, name in zip(
    axs,
    [history_dense, history_cnn],
    ["Dense baseline", "CNN"],
):
    ax.plot(
        history.history["accuracy"],
        label="train",
    )
    ax.plot(
        history.history["val_accuracy"],
        label="validation",
    )

    ax.set_title(f"{name}: accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1)
    ax.legend()

plt.tight_layout()
plt.show()

## 7) Test-set comparison

Both neural networks now receive the same 1,000 test images.

Random chance is roughly 10%.

The classical baseline from `06_01` used only 34 human-designed features. Because that notebook is run independently, its exact score is not hard-coded here—record it when you run the sequence and compare it with the two neural networks.

In [ ]:
dense_test_loss, dense_test_acc = dense_model.evaluate(
    ds_test,
    verbose=0,
)

cnn_test_loss, cnn_test_acc = cnn_model.evaluate(
    ds_test,
    verbose=0,
)

results = pd.DataFrame(
    {
        "model": [
            "Random chance",
            "Dense network on pixels",
            "Small CNN",
        ],
        "test_accuracy": [
            1 / num_classes,
            dense_test_acc,
            cnn_test_acc,
        ],
    }
)

results

### What makes this comparison interesting?

Both neural networks had access to the **same raw pixels**.

So if the CNN performs better, the lesson is not:

> “The CNN had better data.”

It is:

> **The CNN organized the same data in a way that matches the structure of images better.**

That is what an **inductive bias** does: the architecture builds in assumptions about the kind of pattern we expect to matter.

## 8) Look inside the CNN

In `06_00`, a Sobel filter produced one human-designed feature map.

A trained CNN produces many learned feature maps.

Let's take one test image and look at the outputs of the first convolutional layer.

We should **not** expect every map to have an obvious human label.

The point is simply to see that different learned filters respond to different visual patterns.

In [ ]:
feature_model = tf.keras.Model(
    inputs=cnn_model.layers[0].input, # Access the input tensor from the first layer
    outputs=cnn_model.get_layer("conv1").output,
)

example_image = (
    X_test[0].astype(np.float32)
    / 255.0
)

feature_maps = feature_model.predict(
    example_image[None, ...],
    verbose=0,
)[0]

print("Input image shape:", example_image.shape)
print("First convolution output:", feature_maps.shape)

In [ ]:
fig, axs = plt.subplots(
    2,
    5,
    figsize=(12, 6),
)

axs[0, 0].imshow(example_image)
axs[0, 0].set_title(
    f"Input\n{label_names[y_test[0]]}"
)
axs[0, 0].axis("off")

# Show nine of the 32 learned feature maps.
for feature_id, ax in zip(
    range(9),
    axs.flat[1:],
):
    ax.imshow(
        feature_maps[..., feature_id],
        cmap="gray",
    )
    ax.set_title(f"Filter {feature_id}")
    ax.axis("off")

plt.tight_layout()
plt.show()

### What should we see?

Some feature maps may emphasize:

- boundaries;
- bright or dark regions;
- textures;
- color contrasts;
- particular local arrangements.

But unlike our Sobel kernel, these filters were **not named or specified by us**.

They were learned because they helped reduce classification error.

That is the core shift from `06_01` to `06_02`:

\[
\text{human-designed representation}
\rightarrow
\text{learned representation}
\]

## 9) Where does the CNN still fail?

A higher accuracy does not mean the model has solved the task.

We still need to inspect:

- the confusion matrix;
- per-class performance;
- confident errors.

This matters because the next notebook will introduce a stronger source of visual representation: **pretraining**.

In [ ]:
cnn_probabilities = cnn_model.predict(
    ds_test,
    verbose=0,
)

cnn_predictions = cnn_probabilities.argmax(
    axis=1
)

print(
    classification_report(
        y_test,
        cnn_predictions,
        target_names=label_names,
        digits=3,
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    cnn_predictions,
)

fig, ax = plt.subplots(
    figsize=(9, 8),
)

im = ax.imshow(
    cm,
    cmap="Blues",
)

ax.set_xticks(range(num_classes))
ax.set_xticklabels(
    label_names,
    rotation=90,
    fontsize=9,
)

ax.set_yticks(range(num_classes))
ax.set_yticklabels(
    label_names,
    fontsize=9,
)

ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("CNN confusion matrix")

for i in range(num_classes):
    for j in range(num_classes):
        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
            fontsize=8,
            color=(
                "white"
                if cm[i, j] > cm.max() / 2
                else "black"
            ),
        )

plt.colorbar(
    im,
    ax=ax,
    fraction=0.046,
)

plt.tight_layout()
plt.show()

### Inspect confident mistakes

A confident error is useful because it tells us the model did not merely hesitate—it found a pattern that strongly supported the wrong class.

Ask:

> What visual evidence might have pushed the model toward its prediction?

That is a more useful diagnostic question than simply saying “the model was wrong.”

In [ ]:
confidence = cnn_probabilities.max(axis=1)

wrong = np.where(
    cnn_predictions != y_test
)[0]

wrong = wrong[
    np.argsort(
        -confidence[wrong]
    )
]

n_show = min(
    8,
    len(wrong),
)

selected = wrong[:n_show]

fig, axs = plt.subplots(
    2,
    4,
    figsize=(12, 6),
)

for ax in axs.flat:
    ax.axis("off")

for test_pos, ax in zip(
    selected,
    axs.flat,
):
    ax.imshow(
        X_test[test_pos]
    )

    ax.set_title(
        f"true: {label_names[y_test[test_pos]]}\n"
        f"pred: {label_names[cnn_predictions[test_pos]]} "
        f"({confidence[test_pos]:.2f})",
        fontsize=9,
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

## Takeaways

### 1. The dense network saw the pixels but ignored their structure

Flattening preserves the values but removes the architecture's explicit notion of locality.

### 2. Convolution builds the right assumptions into the model

The CNN applies learned local filters across the image using shared weights.

### 3. Feature learning replaces manual feature engineering

In `06_01`, we decided that color histograms and Sobel edges mattered.

Here, gradient descent decides what filters are useful.

### 4. More parameters do not automatically mean a better model

The dense network can contain many more parameters and still perform worse because its architecture is poorly matched to images.

### 5. Learned representations are still imperfect

The confusion matrix and confident mistakes show that some land-cover distinctions remain difficult.

---

## Where the next notebook goes

So far, the CNN started with **random filter weights** and had to learn vision from only 3,500 training images.

That leads to the next question:

> **Why learn visual features from scratch if another network has already learned useful visual representations from millions of images?**

[`06_03_main_transfer.ipynb`](06_03_main_transfer.ipynb) introduces **transfer learning**:

\[
\text{pretrained visual representation}
+
\text{our task}
\rightarrow
\text{adapted classifier}
\]

That is the next step toward modern foundation models.